# BNNR — `analyze` quickstart (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bnnr-team/bnnr/blob/main/examples/analyze_colab_quickstart.ipynb)

**Accuracy tells you *how often* a model is wrong. `bnnr analyze` tells you *why*.**

In ~3 minutes, with no setup on your machine, you get a portable **HTML failure + XAI report** for a PyTorch vision model:
validation metrics, top confused class pairs, the worst misclassifications, and **OptiCAM saliency overlays** showing where the model actually looked.

**Do you need a checkpoint to start? No.** Section 3 quick-trains a tiny ResNet-18 (1 epoch on a CIFAR-10 subset) *for* you, so even with nothing in hand you get a full report. Then:
- **Section 5** — use *your own* checkpoint.
- **Section 6** — use *your own* model **and** *your own* data (the general path).

> Prefer zero install? Open the [sample report](https://raw.githack.com/bnnr-team/bnnr/refs/heads/main/docs/assets/analyze-report-sample.html) (a real MNIST run) in your browser first.

**Tip:** a GPU runtime is faster but not required. `Runtime → Change runtime type → T4 GPU`.

## 1. Install BNNR

Clones the repo (for the example scripts) and installs the package.

In [ ]:
%cd /content
![ -d bnnr ] || git clone --depth=1 https://github.com/bnnr-team/bnnr.git
%cd /content/bnnr
!git pull --rebase
!pip install -e . -q
import os; print('BNNR installed from:', os.getcwd())

## 2. Device check (optional)

CPU is fine for this quick demo (~3 min). A GPU just makes it faster.

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU — running on CPU (fine for the quick demo).')

## 3. Run `analyze` on CIFAR-10 (demo checkpoint)

This quick-trains a ResNet-18 for 1 epoch on a CIFAR-10 subset so there is a checkpoint to analyze,
then runs `analyze_model` to write `report.html` + `analysis_report.json` to `/content/analyze_out`.
Downloads CIFAR-10 (~170 MB) on first run. (Already have a checkpoint? Skip to Section 5.)

In [ ]:
!python examples/classification/torchvision_analyze_cifar10.py --quick --device auto --output-dir /content/analyze_out

## 4. View the report inline

BNNR reports are **self-contained static HTML** — no JavaScript, images embedded as base64.
That means they render inside Colab in an isolated `<iframe>` (the isolation keeps the report's
dark theme from restyling the notebook). The same file opens in any browser, email, or PR attachment.

If the iframe ever looks empty (very large reports), use the download cell below and open it locally.

In [ ]:
import base64
from pathlib import Path
from IPython.display import HTML

html = Path('/content/analyze_out/report.html').read_text(encoding='utf-8')
b64 = base64.b64encode(html.encode()).decode()
HTML(f'<iframe src="data:text/html;base64,{b64}" width="100%" height="720" style="border:1px solid #e5e7eb;border-radius:8px"></iframe>')

In [ ]:
# Guaranteed fallback: download the report and open it in your browser.
from google.colab import files
files.download('/content/analyze_out/report.html')

## 5. Use your own checkpoint (matching architecture)

If your checkpoint is a **ResNet-18 trained on CIFAR-10** (same architecture as the demo), point the
example script straight at it — no retraining. Upload a `.pt`, then run:

In [ ]:
# from google.colab import files; files.upload()  # upload your_checkpoint.pt to /content/bnnr/
# Then run (single line):
# !python examples/classification/torchvision_analyze_cifar10.py --checkpoint /content/bnnr/your_checkpoint.pt --device auto --output-dir /content/analyze_my_ckpt
print('Uncomment after uploading a ResNet-18 / CIFAR-10 checkpoint, then re-run the view cell on /content/analyze_my_ckpt/report.html')

The CLI works the same way when the checkpoint matches a built-in dataset model:
```bash
bnnr analyze --model your_checkpoint.pt --data cifar10 --output report/
# --data also accepts: mnist, fashion_mnist, stl10, or an ImageFolder directory
```
Note: `--data cifar10` (and the script above) expect the **ResNet-18 / built-in CNN** architecture and will
refuse a mismatched checkpoint. For a different architecture or your own dataset, use the general path below.

## 6. Use your own model **and** your own data (general path)

This is the real workflow for any architecture (timm, custom CNN, ViT…) on any dataset. You only need
three things: your `nn.Module` with loaded weights, a `DataLoader` that yields `(image, label, index)`,
and the layer(s) for the saliency maps (usually the last conv block). Adapt this template:

```python
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from bnnr.adapter import SimpleTorchAdapter
from bnnr.analyze import analyze_model
from bnnr.core import BNNRConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1) YOUR model + weights (any nn.Module)
model = MyModel(num_classes=...)
model.load_state_dict(torch.load('my_checkpoint.pt', map_location=device))
model = model.to(device).eval()

# 2) analyze needs (image, label, index). Wrap any (image, label) dataset:
class Indexed(Dataset):
    def __init__(self, base): self.base = base
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        x, y = self.base[i]
        return x, y, i

val_loader = DataLoader(Indexed(your_val_dataset), batch_size=32)

# 3) target layer(s) for OptiCAM/Grad-CAM — the final conv block of YOUR net
#    ResNet: [model.layer4[-1]]   ·   adjust for your architecture
adapter = SimpleTorchAdapter(
    model=model,
    criterion=nn.CrossEntropyLoss(),
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-3),
    target_layers=[model.layer4[-1]],
    device=device,
)

report = analyze_model(
    adapter, val_loader,
    config=BNNRConfig(device=device, task='classification'),
    output_dir='/content/analyze_my_model',
)
report.to_html('/content/analyze_my_model/report.html',
               artifact_root='/content/analyze_my_model', embed_images=True)
print('Saved /content/analyze_my_model/report.html')
```

A complete, runnable version of this pattern (with a torchvision dataset) is
[`examples/classification/torchvision_analyze_cifar10.py`](https://github.com/bnnr-team/bnnr/blob/main/examples/classification/torchvision_analyze_cifar10.py).

## Next steps

- **Docs:** [analyze guide](https://github.com/bnnr-team/bnnr/blob/main/docs/analyze.md) · [getting started](https://github.com/bnnr-team/bnnr/blob/main/docs/getting_started.md)
- **Repo:** [github.com/bnnr-team/bnnr](https://github.com/bnnr-team/bnnr) — if this was useful, a ⭐ helps others find it.

BNNR is MIT-licensed. Built on PyTorch + pytorch-grad-cam.